# Scheduled Batch Inference NotebookThis notebook is intended to be scheduled in Amazon SageMaker Studio to perform batch inference on a trained model. It downloads an input Parquet file from Amazon S3, runs inference using a locally downloaded model artifact, and uploads the predictions as a Parquet file to another Amazon S3 location.

## 1. Configure ParametersWhen scheduling this notebook in SageMaker Studio, override these environment variables to customize input/output locations and the model artifact path. Defaults are provided for local testing.

In [ ]:
import osINPUT_DATA_S3_URI = os.environ.get("INPUT_DATA_S3_URI", "s3://my-input-bucket/path/to/input.parquet")OUTPUT_DATA_S3_URI = os.environ.get("OUTPUT_DATA_S3_URI", "s3://my-output-bucket/path/to/predictions.parquet")MODEL_ARTIFACT_S3_URI = os.environ.get("MODEL_ARTIFACT_S3_URI", "s3://my-model-bucket/path/to/model.tar.gz")LOCAL_WORKDIR = os.environ.get("LOCAL_WORKDIR", "/root/batch-inference")print(f"Input data: {INPUT_DATA_S3_URI}")print(f"Output data: {OUTPUT_DATA_S3_URI}")print(f"Model artifact: {MODEL_ARTIFACT_S3_URI}")print(f"Local workdir: {LOCAL_WORKDIR}")

## 2. Install Dependencies (if needed)Most SageMaker notebook images include the required libraries. Uncomment and adjust the cell below if you need to install additional packages.

In [ ]:
# %%capture
# !pip install --quiet pyarrow pandas s3fs joblib


## 3. Imports and Utility FunctionsThe helper utilities below manage downloading and uploading objects in S3, schema enforcement, and inference execution.

In [ ]:
import loggingimport tarfilefrom datetime import datetimefrom pathlib import Pathfrom typing import Tuplefrom urllib.parse import urlparseimport boto3import joblibimport pandas as pdlogging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")logger = logging.getLogger(__name__)s3_resource = boto3.resource("s3")INPUT_SCHEMA = {    "offer_id": "Int64",    "partner_id": "Int64",    "product_id": "Int64",    "carrier_code": "string",    "flight_number": "Int64",    "origination_code": "string",    "destination_code": "string",    "departure_timestamp": "datetime64[ns]",    "seats_available": "Int64",    "upgrade_type": "string",    "item_count": "Int64",    "usd_base_amount": "float64",    "fare_class": "string",    "from_cabin": "string",    "created_timestamp": "datetime64[ns]",    "multiplier_fare_class": "float64",    "multiplier_loyalty": "float64",    "multiplier_success_history": "float64",    "multiplier_payment_type": "float64",}OUTPUT_COLUMNS = [    "offer_id",    "partner_id",    "product_id",    "carrier_code",    "flight_number",    "origination_code",    "destination_code",    "departure_timestamp",    "upgrade_type",    "accept_prob",    "accept_prob_timestamp",]def parse_s3_uri(s3_uri: str) -> Tuple[str, str]:    parsed = urlparse(s3_uri)    if parsed.scheme != "s3" or not parsed.netloc or not parsed.path:        raise ValueError(f"Invalid S3 URI: {s3_uri}")    bucket = parsed.netloc    key = parsed.path.lstrip("/")    return bucket, keydef ensure_local_directory(path: Path) -> None:    path.mkdir(parents=True, exist_ok=True)def download_s3_object(s3_uri: str, destination: Path) -> Path:    bucket, key = parse_s3_uri(s3_uri)    ensure_local_directory(destination.parent)    logger.info("Downloading %s to %s", s3_uri, destination)    s3_resource.Bucket(bucket).download_file(key, str(destination))    return destinationdef download_and_extract_model(s3_uri: str, workdir: Path) -> Path:    ensure_local_directory(workdir)    archive_path = workdir / "model.tar.gz"    download_s3_object(s3_uri, archive_path)    logger.info("Extracting model archive to %s", workdir)    with tarfile.open(archive_path, "r:gz") as tar:        tar.extractall(path=workdir)    model_paths = list(workdir.rglob("*.joblib")) + list(workdir.rglob("*.pkl"))    if not model_paths:        raise FileNotFoundError("No model file (.joblib or .pkl) found in the extracted archive.")    logger.info("Loaded model file: %s", model_paths[0])    return model_paths[0]def load_parquet(s3_uri: str, workdir: Path) -> pd.DataFrame:    local_path = workdir / "input.parquet"    download_s3_object(s3_uri, local_path)    logger.info("Reading input parquet with schema enforcement")    df = pd.read_parquet(local_path)    for column, dtype in INPUT_SCHEMA.items():        if column not in df.columns:            raise KeyError(f"Expected column '{column}' missing from input data.")        if dtype.startswith("datetime64"):            df[column] = pd.to_datetime(df[column], errors="coerce")        else:            df[column] = df[column].astype(dtype)    return dfdef save_parquet(df: pd.DataFrame, s3_uri: str, workdir: Path) -> None:    local_path = workdir / "predictions.parquet"    ensure_local_directory(local_path.parent)    df.to_parquet(local_path, index=False)    bucket, key = parse_s3_uri(s3_uri)    logger.info("Uploading predictions to %s", s3_uri)    s3_resource.Bucket(bucket).upload_file(str(local_path), key)def load_model(model_path: Path):    logger.info("Loading model from %s", model_path)    return joblib.load(model_path)def run_inference(model, enriched_input: pd.DataFrame) -> pd.DataFrame:    logger.info("Running model inference on %d rows", len(enriched_input))    feature_columns = [        col for col in enriched_input.columns        if col not in {            "offer_id",            "partner_id",            "product_id",            "carrier_code",            "flight_number",            "origination_code",            "destination_code",            "departure_timestamp",            "upgrade_type",        }    ]    predictions = model.predict_proba(enriched_input[feature_columns])[:, 1]    output = enriched_input[[        "offer_id",        "partner_id",        "product_id",        "carrier_code",        "flight_number",        "origination_code",        "destination_code",        "departure_timestamp",        "upgrade_type",    ]].copy()    output["accept_prob"] = predictions.astype("float64")    output["accept_prob_timestamp"] = datetime.utcnow()    return output[OUTPUT_COLUMNS]

## 4. Execute Batch InferenceThe following cell orchestrates the download, inference, and upload steps. Rerun this cell (or schedule the notebook) whenever predictions need to be refreshed.

In [ ]:
workdir = Path(LOCAL_WORKDIR)ensure_local_directory(workdir)input_df = load_parquet(INPUT_DATA_S3_URI, workdir)model_path = download_and_extract_model(MODEL_ARTIFACT_S3_URI, workdir / "model")model = load_model(model_path)predictions_df = run_inference(model, input_df)save_parquet(predictions_df, OUTPUT_DATA_S3_URI, workdir)logger.info("Batch inference complete. %d predictions written.", len(predictions_df))

## 5. (Optional) Cleanup Local ArtifactsIf your job container has limited storage, uncomment the cleanup cell below to remove temporary files after the run.

In [ ]:
# import shutil
# shutil.rmtree(LOCAL_WORKDIR, ignore_errors=True)
